In [ ]:
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime
import numpy as np
import pandas as pd
import sentencepiece as spm
from tqdm import tqdm
import torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
!mkdir -p contrastive_v1 dataset/public_test dataset/private_test

# checkpoint
!gdown 1tCjYZ7S3lymvjac7KdEVzBh3rAeqv8DU -O contrastive_v1/plan_cl_finetune_epoch_30.pt

# public test
!gdown 1b8tAy01TOFE2881Pj9P72l_b6sLk5goF -O dataset/public_test/public_test.zh

# private test
!gdown 1O8bRxQQQU8jve9v7zUNxexj3tVYobXjG -O dataset/private_test/private_test.zh



In [ ]:
import os

CHECKPOINT_PATH = "./contrastive_v1/plan_cl_finetune_epoch_30.pt"
PUBLIC_TEST_PATH = "./dataset/public_test/public_test.zh"
PRIVATE_TEST_PATH = "./dataset/private_test/private_test.zh"
PUBLIC_OUTPUT_PATH = "./public_test.csv"
PRIVATE_OUTPUT_PATH = "./private_test.csv"

BEAM_SIZE = 3
TOP_K = 5
LENGTH_PENALTY = 0.6
MAX_SEQUENCE_LENGTH = 32
DIRECTION_TOKEN = "<2vi>"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Embed the complete model, tokenizer, and decoding stack directly in this notebook.

In [ ]:
import math
import os
import tempfile
from dataclasses import dataclass
from typing import List, Optional, Tuple

import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F


class Config:
    spm_prefix = ""
    d_model = 768
    n_heads = 12
    n_kv_heads = 4
    num_encoder_layers = 8
    num_decoder_layers = 8
    d_ff = 3072
    dropout = 0.01
    max_len = MAX_SEQUENCE_LENGTH
    rope_base = 10000.0
    pad_token = "<pad>"
    bos_token = "<s>"
    eos_token = "</s>"
    unk_token = "<unk>"
    zh_token = "<2zh>"
    vi_token = "<2vi>"


class PlanCLBiCrossConfig(Config):
    pass


class PlanCLFineTuneConfig(Config):
    pass


@dataclass
class InferenceConfig:
    d_model: int = 768
    n_heads: int = 12
    n_kv_heads: int = 4
    num_encoder_layers: int = 8
    num_decoder_layers: int = 8
    d_ff: int = 3072
    dropout: float = 0.01
    max_len: int = MAX_SEQUENCE_LENGTH
    rope_base: float = 10000.0
    pad_token: str = "<pad>"
    bos_token: str = "<s>"
    eos_token: str = "</s>"
    unk_token: str = "<unk>"
    zh_token: str = "<2zh>"
    vi_token: str = "<2vi>"
    spm_prefix: str = ""
    vocab_size: int = 8000
    device: torch.device = DEVICE


def apply_checkpoint_config(base_config: InferenceConfig, checkpoint: dict) -> None:
    ckpt_cfg = checkpoint.get("config")
    if ckpt_cfg is None:
        return
    for field in vars(base_config).keys():
        if hasattr(ckpt_cfg, field):
            setattr(base_config, field, getattr(ckpt_cfg, field))


def materialize_tokenizer(payload: dict) -> Tuple[str, str]:
    if not payload:
        raise ValueError("Tokenizer payload missing or empty in checkpoint.")
    model_bytes = payload.get("model_bytes")
    vocab_bytes = payload.get("vocab_bytes")
    if model_bytes is None or vocab_bytes is None:
        raise ValueError("Tokenizer payload lacks model or vocab bytes.")
    tmp_dir = tempfile.mkdtemp(prefix="spm_from_checkpoint_")
    base_name = os.path.basename(payload.get("prefix", "spm_from_ckpt")) or "spm_from_ckpt"
    prefix = os.path.join(tmp_dir, base_name)
    with open(f"{prefix}.model", "wb") as f:
        f.write(model_bytes)
    with open(f"{prefix}.vocab", "wb") as f:
        f.write(vocab_bytes)
    return prefix, tmp_dir


class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        return self.weight * x / rms


class RoPE(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer("inv_freq", inv_freq)
        self._seq_len_cached = 0
        self._cos_cached = None
        self._sin_cached = None

    def _update(self, seq_len: int, device: torch.device, dtype: torch.dtype):
        needs_refresh = (
            self._cos_cached is None
            or self._sin_cached is None
            or seq_len > self._seq_len_cached
            or self._cos_cached.device != device
            or self._cos_cached.dtype != dtype
        )
        if needs_refresh:
            self._seq_len_cached = seq_len
            position = torch.arange(seq_len, device=device, dtype=dtype)
            freqs = torch.outer(position, self.inv_freq.to(device))
            self._cos_cached = freqs.cos()
            self._sin_cached = freqs.sin()

    def forward(self, x: torch.Tensor, seq_len: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = seq_len or x.size(-2)
        self._update(seq_len, x.device, x.dtype)
        return self._cos_cached[:seq_len], self._sin_cached[:seq_len]


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos
    return torch.stack([rotated_x1, rotated_x2], dim=-1).flatten(-2)


class FFN_SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.linear2 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.d_ff = d_ff

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.linear1(x)
        g, v = h[..., :self.d_ff], h[..., self.d_ff:]
        s = g * torch.sigmoid(g)
        hidden = s * v
        return self.dropout(self.linear2(hidden))


class GroupedQueryAttentionRoPE(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, dropout: float = 0.1, rope_base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
        self.rope = RoPE(self.d_k, base=rope_base)

    def forward(self, q, k, v, key_padding_mask=None, attn_mask=None):
        B, T_q, T_k = q.size(0), q.size(1), k.size(1)
        Q = self.W_q(q).view(B, T_q, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)
        cos_q, sin_q = self.rope(Q, T_q)
        cos_k, sin_k = self.rope(K, T_k)
        Q = apply_rope(Q, cos_q, sin_q)
        K = apply_rope(K, cos_k, sin_k)
        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = torch.matmul(F.softmax(scores, dim=-1), V)
        out = attn.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(self.dropout(out))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttentionRoPE(d_model, n_heads, n_kv_heads, dropout, rope_base)
        self.ln2 = RMSNorm(d_model)
        self.ffn = FFN_SwiGLU(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_pad_mask=None):
        attn_out = self.self_attn(self.ln1(x), self.ln1(x), self.ln1(x), key_padding_mask=src_pad_mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        return x + ffn_out


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttentionRoPE(d_model, n_heads, n_kv_heads, dropout, rope_base)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = GroupedQueryAttentionRoPE(d_model, n_heads, n_kv_heads, dropout, rope_base)
        self.ln3 = RMSNorm(d_model)
        self.ffn = FFN_SwiGLU(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, y, enc_out, tgt_pad_mask=None, tgt_causal_mask=None, src_pad_mask=None):
        y = y + self.dropout(self.self_attn(self.ln1(y), self.ln1(y), self.ln1(y), key_padding_mask=tgt_pad_mask, attn_mask=tgt_causal_mask))
        y = y + self.dropout(self.cross_attn(self.ln2(y), enc_out, enc_out, key_padding_mask=src_pad_mask))
        y = y + self.ffn(self.ln3(y))
        return y


class TransformerModel(nn.Module):
    def __init__(self, config: InferenceConfig, vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.d_model, padding_idx=0)
        self.emb_dropout = nn.Dropout(config.dropout)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(config.d_model, config.n_heads, config.n_kv_heads, config.d_ff, config.dropout, config.rope_base)
            for _ in range(config.num_encoder_layers)
        ])
        self.encoder_final_ln = RMSNorm(config.d_model)
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(config.d_model, config.n_heads, config.n_kv_heads, config.d_ff, config.dropout, config.rope_base)
            for _ in range(config.num_decoder_layers)
        ])
        self.decoder_final_ln = RMSNorm(config.d_model)
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))
        self.emb_scale = math.sqrt(config.d_model)

    def encode(self, src_ids: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb_dropout(self.embedding(src_ids) * self.emb_scale)
        for layer in self.encoder_layers:
            x = layer(x, pad_mask)
        return self.encoder_final_ln(x)

    def decode(self, tgt_ids: torch.Tensor, enc_out: torch.Tensor, tgt_pad_mask: torch.Tensor, tgt_causal_mask: torch.Tensor, src_pad_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb_dropout(self.embedding(tgt_ids) * self.emb_scale)
        for layer in self.decoder_layers:
            x = layer(x, enc_out, tgt_pad_mask, tgt_causal_mask, src_pad_mask)
        return self.decoder_final_ln(x)

    def project(self, hidden: torch.Tensor) -> torch.Tensor:
        return F.linear(hidden, self.embedding.weight, self.output_bias)


@dataclass
class BeamSearchHypothesis:
    tokens: List[int]
    log_prob: float

    def __lt__(self, other):
        return self.log_prob < other.log_prob


def beam_search_decode(model, src_ids, sp_model, config, beam_size=BEAM_SIZE, max_len=None, length_penalty=LENGTH_PENALTY, top_k=TOP_K):
    model.eval()
    batch_size = src_ids.size(0)
    device = src_ids.device
    bos_id = sp_model.piece_to_id(config.bos_token)
    eos_id = sp_model.piece_to_id(config.eos_token)
    pad_id = sp_model.piece_to_id(config.pad_token)
    max_len = max_len or config.max_len
    results = []
    with torch.no_grad():
        src_pad_mask = (src_ids == pad_id)
        enc_out = model.encode(src_ids, src_pad_mask)
        for b in range(batch_size):
            beams = [BeamSearchHypothesis(tokens=[bos_id], log_prob=0.0)]
            finished = []
            curr_enc = enc_out[b:b+1]
            curr_mask = src_pad_mask[b:b+1]
            for _ in range(max_len):
                if not beams:
                    break
                proposals = []
                for hyp in beams:
                    if hyp.tokens[-1] == eos_id:
                        finished.append(hyp)
                        continue
                    tgt_ids = torch.tensor(hyp.tokens, dtype=torch.long, device=device).unsqueeze(0)
                    tgt_pad_mask = (tgt_ids == pad_id)
                    T_tgt = tgt_ids.size(1)
                    causal_mask = torch.triu(torch.ones(T_tgt, T_tgt, dtype=torch.bool, device=device), diagonal=1)
                    dec_out = model.decode(tgt_ids, curr_enc, tgt_pad_mask, causal_mask, curr_mask)
                    logits = model.project(dec_out)
                    log_probs = F.log_softmax(logits[:, -1, :], dim=-1)
                    if top_k is not None and top_k > 0:
                        top_vals, top_idx = torch.topk(log_probs, min(top_k, log_probs.size(-1)))
                        mask = torch.full_like(log_probs, float('-inf'))
                        mask.scatter_(1, top_idx, top_vals)
                        log_probs = mask
                    top_vals, top_idx = torch.topk(log_probs.squeeze(0), beam_size)
                    for val, idx in zip(top_vals.tolist(), top_idx.tolist()):
                        proposals.append(BeamSearchHypothesis(tokens=hyp.tokens + [idx], log_prob=hyp.log_prob + val))
                beams = sorted(proposals, key=lambda h: h.log_prob, reverse=True)[:beam_size]
            finished.extend(beams)
            best = max(finished, key=lambda h: h.log_prob / (len(h.tokens) ** length_penalty)) if finished else BeamSearchHypothesis(tokens=[bos_id, eos_id], log_prob=0.0)
            decoded = best.tokens[1:]
            if eos_id in decoded:
                decoded = decoded[:decoded.index(eos_id)]
            results.append(sp_model.decode(decoded))
    return results


def load_model_and_tokenizer(checkpoint_path: str):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    config = InferenceConfig()
    apply_checkpoint_config(config, checkpoint)
    if "tokenizer" not in checkpoint:
        raise ValueError("Checkpoint must include tokenizer bytes.")
    tokenizer_prefix, _ = materialize_tokenizer(checkpoint["tokenizer"])
    sp_model = spm.SentencePieceProcessor()
    sp_model.Load(f"{tokenizer_prefix}.model")
    vocab_size = sp_model.GetPieceSize()
    config.spm_prefix = tokenizer_prefix
    model = TransformerModel(config, vocab_size).to(config.device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    return model, sp_model, config


def read_dataset(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]


def translate_split(model, sp_model, config, input_path: str, output_path: str, split_name: str) -> pd.DataFrame:
    sentences = read_dataset(input_path)
    rows = []
    model.eval()
    for sentence in tqdm(sentences, desc=f"{split_name} decoding", total=len(sentences)):
        src_with_token = f"{DIRECTION_TOKEN} {sentence}".strip()
        src_ids = sp_model.encode(src_with_token, out_type=int)[:config.max_len]
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=config.device).unsqueeze(0)
        translation = beam_search_decode(
            model,
            src_tensor,
            sp_model,
            config,
            beam_size=BEAM_SIZE,
            max_len=config.max_len,
            length_penalty=LENGTH_PENALTY,
            top_k=TOP_K
        )[0]
        rows.append({"tieng_trung": sentence, "tieng_viet": translation})
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8")
    return df

# 6. Run Public Test Inference & Save `csv`
Load the checkpoint once, decode the public set, and persist the required CSV.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_split_job(split_name: str, input_path: str, output_path: str):
    local_model, local_sp_model, local_config = load_model_and_tokenizer(CHECKPOINT_PATH)
    return translate_split(
        local_model,
        local_sp_model,
        local_config,
        input_path,
        output_path,
        split_name=split_name
    )

jobs = {
    "Public": (PUBLIC_TEST_PATH, PUBLIC_OUTPUT_PATH),
    "Private": (PRIVATE_TEST_PATH, PRIVATE_OUTPUT_PATH)
}

results = {}
with ThreadPoolExecutor(max_workers=2) as executor:
    future_to_name = {
        executor.submit(run_split_job, name, paths[0], paths[1]): name
        for name, paths in jobs.items()
    }
    for future in as_completed(future_to_name):
        split_name = future_to_name[future]
        results[split_name] = future.result()
        print(f"{split_name} split finished -> Saved to {jobs[split_name][1]}")

public_df = results.get("Public")
private_df = results.get("Private")

In [ ]:
print(f"Public CSV: {PUBLIC_OUTPUT_PATH} ({len(public_df)} rows)")
print(f"Private CSV: {PRIVATE_OUTPUT_PATH} ({len(private_df)} rows)")
display(public_df.head())
display(private_df.head())